# 📊 Business Intelligence Datasets – Google Drive Access

This notebook loads BI datasets **directly from Google Drive** — no manual download needed.

It tries three access strategies in order:

| Strategy | When it works |
|----------|---------------|
| **1. Mount Google Drive** | Running in Colab and you accept the permission prompt |
| **2. Direct download URL** | File IDs are filled in `gdrive/catalog.json` |
| **3. GitHub fallback** | Always works — uses the sample CSVs committed in this repo |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Whiteknight54/Business-iInteligence-dataset/blob/main/notebooks/google_drive_access.ipynb)

---
**Google Drive folder:** https://drive.google.com/drive/folders/1hp5KzLCTRb2V7ZjHPsFw_WAjUEU6m53i?usp=sharing

## 0 · Setup

In [ ]:
import json
import os
import sys
import urllib.request
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)

# ── Constants ──────────────────────────────────────────────────────────────
DRIVE_FOLDER_ID = '1hp5KzLCTRb2V7ZjHPsFw_WAjUEU6m53i'
DRIVE_FOLDER_URL = f'https://drive.google.com/drive/folders/{DRIVE_FOLDER_ID}?usp=sharing'

CATALOG_URL = ('https://raw.githubusercontent.com/Whiteknight54/'
               'Business-iInteligence-dataset/main/gdrive/catalog.json')

# Detect whether we are inside Google Colab
IN_COLAB = 'google.colab' in sys.modules
print(f'Running in Colab: {IN_COLAB}')
print(f'Drive folder: {DRIVE_FOLDER_URL}')

## 1 · Load the Dataset Catalog

The catalog is stored in `gdrive/catalog.json` in this repository and lists every dataset,
its expected Drive file ID, and a GitHub fallback URL.

In [ ]:
with urllib.request.urlopen(CATALOG_URL) as resp:
    catalog = json.loads(resp.read().decode())

print(f"Drive folder : {catalog['drive_folder']['name']}")
print(f"Datasets     : {len(catalog['datasets'])}")
for ds in catalog['datasets']:
    has_id = ds['file_id'] != 'REPLACE_WITH_FILE_ID'
    status = '✅ file_id set' if has_id else '⚠️  file_id missing (will use GitHub fallback)'
    print(f"  • {ds['name']:25s}  {status}")

## 2 · Mount Google Drive (Colab only)

If you are running in Colab, the cell below mounts your Google Drive so files can be
read as local paths.  A permission dialog will appear — click **Connect to Google Drive**
and select the account that owns the shared folder.

> **Outside Colab** this cell is skipped automatically.

In [ ]:
DRIVE_MOUNTED = False
DRIVE_ROOT = '/content/drive/MyDrive'

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_MOUNTED = os.path.isdir(DRIVE_ROOT)
    print(f'Drive mounted: {DRIVE_MOUNTED}')
    if DRIVE_MOUNTED:
        print(f'Drive root contents: {os.listdir(DRIVE_ROOT)[:10]}')
else:
    print('Not in Colab — skipping Drive mount.')

## 3 · Smart Dataset Loader

The loader below tries each strategy in order and returns the first one that succeeds.

In [ ]:
def load_dataset(ds_entry: dict, drive_subfolder: str = 'Business Intelligence Datasets',
                 **read_kwargs) -> pd.DataFrame:
    """
    Load a dataset using the best available strategy.

    Strategy 1 – mounted Drive path
    Strategy 2 – direct Drive download URL (requires file_id in catalog)
    Strategy 3 – GitHub raw fallback
    """
    name = ds_entry['name']
    filename = ds_entry['filename']
    folder = ds_entry['folder']
    file_id = ds_entry['file_id']
    download_url = ds_entry['download_url']
    fallback_url = ds_entry['github_fallback']

    # Strategy 1: mounted Drive
    if DRIVE_MOUNTED:
        drive_path = os.path.join(DRIVE_ROOT, drive_subfolder, folder, filename)
        if os.path.exists(drive_path):
            print(f'[{name}] Loading from mounted Drive: {drive_path}')
            return pd.read_csv(drive_path, **read_kwargs)
        else:
            print(f'[{name}] Drive path not found: {drive_path}')

    # Strategy 2: direct download URL
    if file_id != 'REPLACE_WITH_FILE_ID':
        print(f'[{name}] Loading via Drive download URL (file_id={file_id})')
        try:
            return pd.read_csv(download_url, **read_kwargs)
        except Exception as exc:
            print(f'[{name}] Drive download failed: {exc}')

    # Strategy 3: GitHub fallback
    print(f'[{name}] Using GitHub fallback: {fallback_url}')
    return pd.read_csv(fallback_url, **read_kwargs)


print('Loader ready ✓')

## 4 · Load All Datasets

In [ ]:
# Build a lookup by dataset name for convenience
ds_map = {ds['name']: ds for ds in catalog['datasets']}

sales        = load_dataset(ds_map['Sales Data'],        parse_dates=['order_date'])
customers    = load_dataset(ds_map['Customer Data'],     parse_dates=['first_purchase_date', 'last_active_date'])
finance      = load_dataset(ds_map['Financial Data'])
marketing    = load_dataset(ds_map['Marketing Data'],    parse_dates=['start_date', 'end_date'])
supply_chain = load_dataset(ds_map['Supply Chain Data'], parse_dates=['order_date'])

print('\nDataset shapes:')
for label, df in [('Sales', sales), ('Customers', customers),
                  ('Finance', finance), ('Marketing', marketing),
                  ('Supply Chain', supply_chain)]:
    print(f'  {label:15s}: {df.shape}')

## 5 · Quick Exploration

In [ ]:
print('── Sales ──')
display(sales.head(3))
print('\nRevenue by region:')
display(sales.groupby('region')['revenue'].sum().sort_values(ascending=False))

In [ ]:
print('── Customers ──')
display(customers.head(3))
print('\nSegment breakdown:')
display(customers['segment'].value_counts())

In [ ]:
print('── Finance ──')
display(finance[['period', 'total_revenue', 'gross_profit', 'net_income', 'net_margin_pct']])

In [ ]:
print('── Marketing ──')
display(marketing[['campaign_name', 'channel', 'spend_usd', 'revenue_generated', 'roas']].head(5))

In [ ]:
print('── Supply Chain ──')
display(supply_chain[['order_id', 'supplier_name', 'product_name', 'lead_time_days', 'on_time_delivery']].head(5))

## 6 · Sample Visualisations

In [ ]:
fig, axes = plt.subplots(1, 2)

# Revenue by region
sales.groupby('region')['revenue'].sum().sort_values().plot(
    kind='barh', ax=axes[0], title='Revenue by Region', color='steelblue')
axes[0].set_xlabel('Revenue (USD)')

# Revenue by category
sales.groupby('category')['revenue'].sum().sort_values().plot(
    kind='barh', ax=axes[1], title='Revenue by Category', color='darkorange')
axes[1].set_xlabel('Revenue (USD)')

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.plot(finance['period'], finance['total_revenue'],  marker='o', label='Revenue')
ax.plot(finance['period'], finance['gross_profit'],   marker='s', label='Gross Profit')
ax.plot(finance['period'], finance['net_income'],     marker='^', label='Net Income')
ax.set_title('Quarterly Financial Performance')
ax.set_ylabel('USD')
ax.legend()
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
marketing.groupby('channel')['roas'].mean().sort_values(ascending=False).plot(
    kind='bar', title='Average ROAS by Channel', color='mediumseagreen')
plt.ylabel('ROAS')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

---
## 7 · Tips: Working with the Drive Folder

### Finding a file ID
1. Open the file in Google Drive.
2. Right-click → **Get link** or look at the URL — the ID is the long string after `/file/d/` or `?id=`.
3. Paste it into `gdrive/catalog.json` under the correct `file_id` key and commit.

### Sharing the folder
For `pd.read_csv` downloads to work without authentication:
- Right-click the Drive folder → **Share** → **Anyone with the link** → **Viewer**.

### Reading a specific Drive file by ID (no catalog needed)
```python
file_id = 'YOUR_FILE_ID'
df = pd.read_csv(f'https://drive.google.com/uc?export=download&id={file_id}')
```

### Listing all files in the mounted Drive folder
```python
import os
folder = '/content/drive/MyDrive/Business Intelligence Datasets'
for root, dirs, files in os.walk(folder):
    level = root.replace(folder, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    for f in files:
        print(f'{indent}  {f}')
```